In [ ]:
!pip install transformers rouge_score torch evaluate datasets gensim

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 43.5 MB/s eta 0:00:00
  

In [ ]:
import torch
import pandas as pd
from transformers import PegasusForConditionalGeneration, PegasusTokenizer, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from rouge_score import rouge_scorer
from sklearn.metrics import precision_score

# Load Dataset
def load_data(file_path):
    df = pd.read_csv(file_path)
    df = df[['article', 'highlights']].dropna()
    return df

# Extract Topics using LDA
def extract_topics_lda(documents, num_topics=10):
    vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
    dtm = vectorizer.fit_transform(documents)

    lda_model = LatentDirichletAllocation(n_components=num_topics, random_state=42)
    lda_topics = lda_model.fit_transform(dtm)

    topic_words = vectorizer.get_feature_names_out()
    topic_strings = [" ".join([topic_words[i] for i in topic.argsort()[-5:]]) for topic in lda_topics]
    return topic_strings

# Preprocess Data
def preprocess_data(train_file, val_file, test_file):
    train_df = load_data(train_file)
    val_df = load_data(val_file)
    test_df = load_data(test_file)

    train_df['topics'] = extract_topics_lda(train_df['article'].tolist())
    val_df['topics'] = extract_topics_lda(val_df['article'].tolist())
    test_df['topics'] = extract_topics_lda(test_df['article'].tolist())

    return train_df, val_df, test_df

# Tokenize Data
def tokenize_data(df, tokenizer):
    text_inputs = (df['topics'] + " " + df['article']).tolist()  # Convert to list
    target_texts = df['highlights'].tolist()

    inputs = tokenizer(text_inputs, truncation=True, padding='max_length', max_length=512)
    targets = tokenizer(target_texts, truncation=True, padding='max_length', max_length=128)

    return Dataset.from_dict({
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask'],
        'labels': targets['input_ids']
    })

# Compute ROUGE and Precision Scores
def compute_metrics(predictions, references):
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = [scorer.score(pred, ref) for pred, ref in zip(predictions, references)]

    avg_rouge = {
        'rouge1': sum([s['rouge1'].fmeasure for s in scores]) / len(scores),
        'rouge2': sum([s['rouge2'].fmeasure for s in scores]) / len(scores),
        'rougeL': sum([s['rougeL'].fmeasure for s in scores]) / len(scores)
    }

    # Precision Calculation (word-level matching)
    y_pred = [" ".join(pred.split()[:len(ref.split())]) for pred, ref in zip(predictions, references)]
    precision = precision_score(
        [set(ref.split()) for ref in references],
        [set(pred.split()) for pred in y_pred],
        average='micro'
    )

    return avg_rouge, precision

# Load dataset
train_file = "train_sampled.csv"
val_file = "validation_sampled.csv"
test_file = "test_sampled.csv"

train_df, val_df, test_df = preprocess_data(train_file, val_file, test_file)

# Load PEGASUS Tokenizer
tokenizer = PegasusTokenizer.from_pretrained("google/pegasus-cnn_dailymail")

# Tokenize data
train_dataset = tokenize_data(train_df, tokenizer)
val_dataset = tokenize_data(val_df, tokenizer)

# Load Model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = PegasusForConditionalGeneration.from_pretrained("google/pegasus-cnn_dailymail").to(device)

# Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    save_total_limit=1,
    logging_dir="./logs",
    fp16=True if torch.cuda.is_available() else False,
)

# Train Model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: chauhanaditya3112 (chauhanaditya3112-indian-institute-of-technology-kanpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Epoch,Training Loss,Validation Loss
1,0.699900,0.892170
2,0.553100,0.911535
3,0.507400,0.932561


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:2758: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 128, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=4308, training_loss=0.9045634433530279, metrics={'train_runtime': 3332.4473, 'train_samples_per_second': 2.585, 'train_steps_per_second': 1.293, 'total_flos': 1.2443478499786752e+16, 'train_loss': 0.9045634433530279, 'epoch': 3.0})

In [ ]:
generated_summaries[1]

'The federal government will give Shoshana Hebshi $40,000 as compensation for being humiliated on the 10th anniversary of the 9/11 terrorist attacks . Armed agents forced her from a plane at Detroit Metropolitan Airport, made her undress during a search and held her for hours . Frontier Airlines, the Transportation Security Administration and Wayne County Airport Authority were named in the federal lawsuit . Hebshi was traveling home after visiting a sister in California when was removed from the Frontier Airlines flight after it landed Sept. 11, 2011 . She was seated next to two Indian-American men, whom crew members had said spent a lot of time in'

In [ ]:
import torch
from rouge_score import rouge_scorer
from sklearn.metrics import precision_score

# Prepare test data
test_texts = (test_df['topics'] + " " + test_df['article']).tolist()
test_highlights = test_df['highlights'].tolist()

# Generate summaries
generated_summaries = []
for text in test_texts:
    inputs = tokenizer(text, truncation=True, padding='max_length', max_length=512, return_tensors="pt").to(device)
    output = model.generate(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'], max_length=128, num_beams=5)
    generated_summary = tokenizer.decode(output[0], skip_special_tokens=True)
    generated_summaries.append(generated_summary)

# Compute ROUGE scores
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}

for ref, pred in zip(test_highlights, generated_summaries):
    scores = scorer.score(pred, ref)
    rouge_scores['rouge1'].append(scores['rouge1'].fmeasure)
    rouge_scores['rouge2'].append(scores['rouge2'].fmeasure)
    rouge_scores['rougeL'].append(scores['rougeL'].fmeasure)

# Calculate average ROUGE scores
avg_rouge1 = sum(rouge_scores['rouge1']) / len(rouge_scores['rouge1'])
avg_rouge2 = sum(rouge_scores['rouge2']) / len(rouge_scores['rouge2'])
avg_rougeL = sum(rouge_scores['rougeL']) / len(rouge_scores['rougeL'])

# Calculate word-level precision score
total_predicted = sum(len(pred.split()) for pred in generated_summaries)
total_correct = sum(len(set(pred.split()) & set(ref.split())) for pred, ref in zip(generated_summaries, test_highlights))

precision = total_correct / total_predicted if total_predicted > 0 else 0.0

# Print evaluation results
print(f"Average ROUGE-1 Score: {avg_rouge1:.4f}")
print(f"Average ROUGE-2 Score: {avg_rouge2:.4f}")
print(f"Average ROUGE-L Score: {avg_rougeL:.4f}")
print(f"Word-level Precision Score: {precision:.4f}")


Average ROUGE-1 Score: 0.4405
Average ROUGE-2 Score: 0.2064
Average ROUGE-L Score: 0.3101
Word-level Precision Score: 0.3381


In [ ]:
generated_summaries[1]

'The federal government will give Shoshana Hebshi $40,000 as compensation for being humiliated on the 10th anniversary of the 9/11 terrorist attacks . Armed agents forced her from a plane at Detroit Metropolitan Airport, made her undress during a search and held her for hours . Frontier Airlines, the Transportation Security Administration and Wayne County Airport Authority were named in the federal lawsuit . Hebshi was traveling home after visiting a sister in California when was removed from the Frontier Airlines flight after it landed Sept. 11, 2011 . She was seated next to two Indian-American men, whom crew members had said spent a lot of time in'

In [ ]:
test_highlights[1]

"The federal government will give Shoshana Hebshi $40,000 as compensation for being ethnically profiled .\nHebshi, who has a Jewish mother and Saudi Arabian father, has said she was discriminated against based on her dark complexion .\nHebshi was detained along with two Indian men she was seated next to .\n'People do not forfeit their constitutional rights when they step onto an airplane,' said ACLU attorney Rachel Goodman ."